# Cardiac digital twins: diffusion maps and PLoM

This walkthrough uses the packaged workflow adapted from the original research notebook. Install `pip install -e ".[notebook]"` from the repository root first. Data are supplied separately; see `data/README.md`. Start with the small execution check below, then use `Config()` for the original Monte Carlo settings.


## Paper reproduction
Run `cardiac-dt-reproduce --data-dir ../data --output-dir results/paper` from the repository root to recompute both regimes and check Tables 1 and 3. The checks reproduce the revision's numerical results, but the unresolved manuscript and data-provenance issues in `docs/PAPER_AUDIT.md` still require reconciliation.


In [ ]:
from pathlib import Path
from cardiac_dt.workflow import Config, run

# Jupyter usually starts this notebook's kernel in the notebooks directory.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Existing local research arrays; for a public clone, set your data directory.
DATA_DIR = ROOT.parent / "data"
OUTPUT_DIR = ROOT / "results" / "notebook-quick"
config = Config(case="nonarrhythmia", n_mc=2, transient_steps=5, n_jobs=1)


## Fit and generate

The short configuration checks execution; it does not reproduce the paper. Use a new output directory on each run. The full embedding is fitted before the lifting split, so lifting test MSE is a transductive reconstruction diagnostic. Set `validate=True` only when the class-specific reference files are available.


In [ ]:
result = run(DATA_DIR, OUTPUT_DIR, config, validate=False)
result["metrics"]


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUTPUT_DIR / "generated_responses.png")))


## Optional conditional sampling

This reproduces the exploratory sampler operations in the source notebook. It is separate from the standardized conditional second-moment estimator used for Table 1. This sampler retains the raw-input metric from the source code; see `docs/PAPER_AUDIT.md`. Effective sample size indicates how strongly the query is supported by nearby generated inputs.


In [ ]:
import numpy as np
from cardiac_dt.data import load_data
from cardiac_dt.plom import ConditionalSampler

inputs, outputs = load_data(DATA_DIR)
query = inputs[result["source_indices"][0]]
sampler = ConditionalSampler(random_state=1).fit(W=result["inputs"], R=result["outputs"], Q=result["generated_coordinates"].T)
bandwidth = 3.0 * sampler.silverman_bandwidth()
response_mean = sampler.conditional_expectation(query, bandwidth=bandwidth)
response_samples = sampler.conditional_samples(query, bandwidth=bandwidth, n_samples=30, random_state=1)
q_mean = sampler.conditional_mean_q(query, bandwidth=bandwidth)
q_covariance = sampler.conditional_cov_q(query, bandwidth=bandwidth)
q_samples = sampler.sample_q_given_w(query, bandwidth=bandwidth, n_draws=30, random_state=1)
q_gaussian_samples = sampler.sample_q_given_w_local_gaussian(query, bandwidth=bandwidth, n_draws=30, random_state=1)
print("Effective sample size:", sampler.effective_sample_size(query, bandwidth=bandwidth))


In [ ]:
import matplotlib.pyplot as plt

time_ms = np.linspace(0, config.duration_ms, outputs.shape[1], endpoint=False)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time_ms, np.asarray(response_samples).T, color="royalblue", alpha=0.15)
ax.plot(time_ms, response_mean, color="black", label="Conditional mean")
ax.set(xlabel="Time (ms)", ylabel="Voltage (mV)")
ax.legend()
plt.show()


## Full notebook settings

Use `Config(case="nonarrhythmia")` or `Config(case="arrhythmia")` for 100 Monte Carlo trajectories and 50 transient steps. Reference comparison requires the corresponding files described in the README. The revised-paper defaults apply a -90 mV lower floor. The paper reproduction uses the revision reference arrays; their simulator provenance is unresolved. See the paper audit.
